# Проверка карточек моделей — compact

Компактная версия без QA-контура.  
Единица отчёта: `model_ver_sid`.  
`1` во флаге = дефект, `0` = дефект не выявлен.

In [ ]:
import os
import re
import sys
from datetime import date, datetime
from functools import reduce
from pathlib import Path

SPARK_CLIENT = Path("/usr/sdp/current/spark3.5.1-client")
if SPARK_CLIENT.exists():
    os.environ.setdefault("SPARK_MAJOR_VERSION", "3.5.1")
    os.environ.setdefault("SPARK_HOME", str(SPARK_CLIENT))
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    sys.path.insert(0, str(SPARK_CLIENT / "python"))
    py4j = sorted((SPARK_CLIENT / "python/lib").glob("py4j-*-src.zip"))
    if py4j:
        sys.path.insert(0, str(py4j[-1]))

from pyspark import SparkConf
from pyspark.sql import SparkSession, Window, functions as F
from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
AS_OF = date.today().isoformat()
OUTPUT_FILE = Path.cwd() / "model_card_audit.xlsx"

EMPTY = {"", "null", "none", "nan", "-", "[]", "{}"}
NA = {"$not_applicable$", "not applicable", "не применимо", "неприменимо", "n/a", "na"}
NO_RESULT = {"отсутствует", *NA}
LOCATION_RE = (
    r"(?i)(?:"
    r"(?:https?|hdfs|s3a?|abfss?|wasbs?|file|dbfs)://[^\s<>]+"
    r"|\\\\[^\s\\]+\\[^\s]+"
    r"|(?:^|\s)/(?:[^\s/]+/)+[^\s]*"
    r"|(?:^|\s)[a-z]:\\[^\s]+"
    r"|\b[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*(?:\.[a-z_][a-z0-9_]*)?\b"
    r")"
)

FLAGS = [
    "F01_MODEL_CODE", "F02_MODEL_NAME", "F03_MODEL_CODE_DUP",
    "F04_VERSION_NUM", "F05_VERSION_NAME", "F06_DEV_START",
    "F07_DEV_END", "F08_EXPL_START", "F09_OWNER",
    "F10_DEVELOPER", "F11_BUSN_PROCESS", "F12_SIGNIFICANCE",
    "F13_DEV_REPORT", "F14_BITBUCKET_CODE", "F15_VALIDATION_RESULT",
    "F16_MONITOR_RESULT", "F17_VALIDATION_DATA", "F18_MONITOR_DATA",
    "F19_SIGNIFICANCE_MISMATCH",
]

print("Дата среза:", AS_OF)
print("Файл:", OUTPUT_FILE)


## Spark

In [ ]:
spark = SparkSession.getActiveSession()
if spark is None:
    conf = (
        SparkConf()
        .setAppName("model_card_audit")
        .setMaster("yarn")
        .set("spark.executor.cores", "2")
        .set("spark.executor.memory", "6g")
        .set("spark.executor.memoryOverhead", "1g")
        .set("spark.driver.maxResultSize", "4g")
        .set("spark.shuffle.service.enabled", "true")
        .set("spark.dynamicAllocation.enabled", "true")
        .set("spark.dynamicAllocation.initialExecutors", "4")
        .set("spark.dynamicAllocation.maxExecutors", "12")
        .set("spark.port.maxRetries", "150")
    )
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)


## Функции

In [ ]:
def norm(c):
    return F.lower(F.trim(F.regexp_replace(c.cast("string"), "\u00a0", " ")))


def filled(c):
    v = norm(c)
    empty_array = v.rlike(r'^\[\s*(?:null|""|\s|,)*\s*\]$')
    return F.coalesce((~v.isin(*EMPTY)) & (~empty_array), F.lit(False))


def required(c):
    return filled(c) & F.coalesce(~norm(c).isin(*NO_RESULT), F.lit(False))


def truthy(c):
    return F.coalesce(norm(c).isin("true", "1", "да"), F.lit(False))


def is_na(c):
    return F.coalesce(norm(c).isin(*NA), F.lit(False))


def has_location(c):
    return filled(c) & F.coalesce(c.cast("string").rlike(LOCATION_RE), F.lit(False))


def valid_significance(c):
    return F.coalesce(F.upper(F.trim(c.cast("string"))).isin("A", "B", "C", "D", "E"), F.lit(False))


def read_state(
    table,
    keys,
    fields,
    as_of,
    current=True,
    nullable_keys=(),
    start_col="start_dt",
    end_col="end_dt",
    where=None,
):
    """Последнее состояние по ключу; умеет date- и datetime-validity."""
    full = f"{SOURCE_DB}.{table}"
    if not spark.catalog.tableExists(full):
        raise RuntimeError(f"Не найдена таблица: {full}")

    raw = spark.table(full)
    names = [c.lower() for c in raw.columns]
    if len(names) != len(set(names)):
        raise RuntimeError(f"{full}: неоднозначные имена столбцов")
    raw = raw.toDF(*names)

    need = set(keys + fields + [start_col])
    missing = sorted(need - set(raw.columns))
    if missing:
        raise RuntimeError(f"{full}: отсутствуют поля: {', '.join(missing)}")

    # where применяем к сырой таблице ДО select().
    # Это важно для случаев, когда фильтр использует служебные поля,
    # которые не входят в payload (например ENT_TYPE_NAME и ENT_PARAM_CHG_SID).
    df = raw
    if where is not None:
        df = df.filter(where)

    meta = [
        c for c in (
            start_col,
            end_col,
            "ctl_datechange",
            "ctl_loading",
            "ctl_action",
        )
        if c in df.columns
    ]
    payload = list(dict.fromkeys(keys + fields))
    df = df.select(*dict.fromkeys(payload + meta))

    for c in df.columns:
        if c.endswith("_sid"):
            df = df.withColumn(c, F.col(c).cast("string"))

    timestamp_grain = start_col.endswith("_dttm")
    bound = (
        F.to_timestamp(F.lit(as_of + " 23:59:59"))
        if timestamp_grain
        else F.lit(as_of).cast("date")
    )
    cast_type = "timestamp" if timestamp_grain else "date"

    df = df.filter(F.col(start_col).cast(cast_type) <= bound)
    if current and end_col in df.columns:
        end = F.col(end_col).cast(cast_type)
        df = df.filter(end.isNull() | (end >= bound))

    mandatory = [k for k in keys if k not in set(nullable_keys)]
    if mandatory:
        bad = reduce(lambda a, b: a | b, [~required(F.col(k)) for k in mandatory], F.lit(False))
        if df.filter(bad).limit(1).count():
            raise RuntimeError(f"{full}: пустой обязательный ключ {mandatory}")

    order = [F.col(start_col).desc_nulls_last()]
    if "ctl_datechange" in df.columns:
        order.append(F.col("ctl_datechange").desc_nulls_last())
    if "ctl_loading" in df.columns:
        order.append(F.col("ctl_loading").cast("long").desc_nulls_last())

    w = Window.partitionBy(*keys).orderBy(*order)
    ranked = df.withColumn("__r", F.dense_rank().over(w)).filter(F.col("__r") == 1)

    state_cols = payload + (["ctl_action"] if "ctl_action" in ranked.columns else [])
    state = ranked.select(*state_cols).dropDuplicates()

    ambiguous = state.groupBy(*keys).count().filter(F.col("count") > 1)
    if ambiguous.limit(1).count():
        examples = [r.asDict() for r in ambiguous.select(*keys).limit(5).collect()]
        raise RuntimeError(f"{full}: неоднозначное последнее состояние; примеры: {examples}")

    if "ctl_action" in state.columns:
        state = state.filter(F.coalesce(F.upper(F.trim(F.col("ctl_action"))) != "D", F.lit(True)))

    return state.select(*payload)


def marker(df, key, condition, name):
    return df.filter(condition).select(key).distinct().withColumn(name, F.lit(1))


def pick_active_prom(as_of):
    rows = read_state(
        "t_model_ver_anlt_dtl",
        ["model_ver_sid", "model_ver_prom_sid"],
        [
            "model_ver_prom_expl_flag",
            "model_ver_not_valid_flag",
            "model_ver_busn_proc_code",
            "model_ver_prom_actual_implm_first_dttm",
            "model_ver_prom_actual_implm_last_dttm",
            "model_ver_prom_crtn_dttm",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_valid_last_rslt_name",
        ],
        as_of,
        nullable_keys=["model_ver_prom_sid"],
    ).filter(truthy(F.col("model_ver_prom_expl_flag")))

    order = [
        F.col("model_ver_prom_actual_implm_last_dttm").desc_nulls_last(),
        F.col("model_ver_prom_actual_implm_first_dttm").desc_nulls_last(),
        F.col("model_ver_prom_crtn_dttm").desc_nulls_last(),
    ]
    w = Window.partitionBy("model_ver_sid").orderBy(*order)
    return rows.withColumn("__r", F.dense_rank().over(w)).filter(F.col("__r") == 1).drop("__r")


def code_links(top_prom, as_of):
    # Основной источник: UI-реквизит "Ссылка на код модели в Bitbucket".
    direct = read_state(
        "t_ent_param_chg",
        ["ent_sid"],
        ["ent_param_chg_val"],
        as_of,
        start_col="start_dttm",
        end_col="end_dttm",
        where=(
            (F.upper(F.trim(F.col("ent_type_name"))) == "MODEL_VERSION")
            & (F.upper(F.trim(F.col("ent_param_chg_sid"))) == "MODEL_VERSION_MODEL_CODE_BITBUCKET")
        ),
    ).select(
        F.col("ent_sid").alias("model_ver_sid"),
        F.col("ent_param_chg_val").alias("__direct_code_link"),
    )

    # Fallback: commit-link актуальной промышленной версии.
    prom = read_state(
        "t_model_ver_prom",
        ["model_ver_prom_sid"],
        ["model_ver_sid", "model_ver_prom_repstry_commit_link_txt"],
        as_of,
    )
    fallback = (
        top_prom.select("model_ver_sid", "model_ver_prom_sid").distinct()
        .join(prom, ["model_ver_sid", "model_ver_prom_sid"], "left")
        .select("model_ver_sid", "model_ver_prom_repstry_commit_link_txt")
        .dropDuplicates()
    )

    ambiguous = fallback.groupBy("model_ver_sid").count().filter(F.col("count") > 1)
    if ambiguous.limit(1).count():
        raise RuntimeError("Для актуальной промверсии найдены разные commit-link")

    return (
        direct.join(fallback, "model_ver_sid", "full")
        .withColumn(
            "bitbucket_link",
            F.coalesce(
                F.when(required(F.col("__direct_code_link")), F.col("__direct_code_link")),
                F.when(required(F.col("model_ver_prom_repstry_commit_link_txt")),
                       F.col("model_ver_prom_repstry_commit_link_txt")),
            ),
        )
        .select("model_ver_sid", "bitbucket_link")
    )


def monitor_data(samples, results, links, kind):
    rsid = f"montrg_{kind}_rslt_sid"
    msid = f"montrg_{kind}_sid"
    rname = f"montrg_{kind}_rslt_name"
    no_sample = f"montrg_{kind}_rslt_valid_not_sample_data_flag"

    sample_ok = marker(
        samples, rsid,
        required(F.col(rsid)) & has_location(F.col("sample_data_addtnl_info_txt")),
        "__sample_ok",
    )

    by_monitor = (
        results.join(sample_ok, rsid, "left")
        .filter(required(F.col(rname)))
        .withColumn(
            "__data_ok",
            (truthy(F.col(no_sample)) | (F.coalesce(F.col("__sample_ok"), F.lit(0)) == 1)).cast("int"),
        )
        .groupBy(msid).agg(F.max("__data_ok").alias("__data_ok"))
    )

    return (
        links.join(by_monitor, msid, "left")
        .select(
            F.col("model_ver_sid").alias(f"__{kind}_model_ver_sid"),
            F.col(msid).alias(f"__{kind}_monitor_sid"),
            F.coalesce(F.col("__data_ok"), F.lit(0)).alias(f"__{kind}_monitor_data_ok"),
        )
    )


## Расчёт

In [ ]:
def build_report(as_of):
    datetime.strptime(as_of, "%Y-%m-%d").date()

    models = read_state("t_model", ["model_sid"], ["model_code", "model_name"], as_of)

    versions = read_state(
        "t_model_ver", ["model_ver_sid"],
        [
            "model_sid", "model_ver_num", "model_ver_name",
            "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_lvl_name",
            "model_ver_owner_block_name", "model_ver_owner_dprtmt_name", "model_ver_owner_cont_name",
            "model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_dev_dprtmt_cont_name",
            "model_ver_dev_start_fact_dttm", "model_ver_dev_end_fact_dttm",
            "model_ver_dev_report_sid", "model_ver_montrg_last_rslt_name",
            "model_ver_montrg_last_sid",
        ], as_of,
    )

    top_prom = pick_active_prom(as_of)

    analytics_cols = [
        "model_ver_sid", "model_ver_not_valid_flag", "model_ver_busn_proc_code",
        "model_ver_prom_actual_implm_first_dttm",
        "model_ver_signfcnt_ctgry_code", "model_ver_valid_last_rslt_name",
    ]
    analytics = top_prom.select(*analytics_cols).dropDuplicates()

    ambiguous = analytics.groupBy("model_ver_sid").count().filter(F.col("count") > 1)
    if ambiguous.limit(1).count():
        raise RuntimeError("Неоднозначная актуальная промышленная строка в T_MODEL_VER_ANLT_DTL")

    code = code_links(top_prom, as_of)

    base = (
        versions.alias("v")
        .join(analytics.alias("a"), "model_ver_sid", "inner")
        .select(
            "model_ver_sid",
            F.col("v.model_sid").alias("model_sid"),
            *[F.col(f"v.{c}").alias(c) for c in [
                "model_ver_num", "model_ver_name",
                "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_lvl_name",
                "model_ver_owner_block_name", "model_ver_owner_dprtmt_name", "model_ver_owner_cont_name",
                "model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_dev_dprtmt_cont_name",
                "model_ver_dev_start_fact_dttm", "model_ver_dev_end_fact_dttm",
                "model_ver_dev_report_sid", "model_ver_montrg_last_rslt_name",
                "model_ver_montrg_last_sid",
            ]],
            F.col("a.model_ver_signfcnt_ctgry_code").alias("__anlt_signfcnt"),
            F.col("a.model_ver_not_valid_flag").alias("model_ver_not_valid_flag"),
            F.col("a.model_ver_busn_proc_code").alias("model_ver_busn_proc_code"),
            F.col("a.model_ver_prom_actual_implm_first_dttm").alias("expl_start_dttm"),
            F.col("a.model_ver_valid_last_rslt_name").alias("model_ver_valid_last_rslt_name"),
        )
        .join(models, "model_sid", "left")
        .join(code, "model_ver_sid", "left")
    )

    dup_codes = (
        models.filter(required(F.col("model_code")))
        .withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .groupBy("__code")
        .agg(F.countDistinct("model_sid").alias("__code_count"))
    )
    base = (
        base.withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .join(dup_codes, "__code", "left")
    )

    # ---------- Валидация ----------
    validations = read_state(
        "t_valid", ["valid_sid"],
        ["model_ver_sid", "valid_rslt_name", "valid_end_fact_dttm", "valid_not_sample_data_flag"],
        as_of, current=False,
    ).filter(required(F.col("model_ver_sid")))

    completed = validations.filter(
        required(F.col("valid_end_fact_dttm"))
        & (F.col("valid_end_fact_dttm").cast("date") <= F.lit(as_of).cast("date"))
    )

    samples = read_state(
        "t_sample_data", ["sample_data_sid"],
        [
            "model_ver_sid", "valid_sid", "montrg_manual_rslt_sid", "montrg_auto_rslt_sid",
            "sample_data_type_name", "sample_data_addtnl_info_txt",
        ],
        as_of, current=False,
    )

    valid_direct = marker(
        samples, "valid_sid",
        required(F.col("valid_sid")) & has_location(F.col("sample_data_addtnl_info_txt")),
        "__valid_direct",
    )
    valid_version = marker(
        samples, "model_ver_sid",
        required(F.col("model_ver_sid"))
        & norm(F.col("sample_data_type_name")).contains("валидац")
        & has_location(F.col("sample_data_addtnl_info_txt")),
        "__valid_version",
    )

    validation_stats = (
        completed.join(valid_direct, "valid_sid", "left")
        .join(valid_version, "model_ver_sid", "left")
        .withColumn("__result", required(F.col("valid_rslt_name")).cast("int"))
        .withColumn(
            "__data",
            (
                required(F.col("valid_rslt_name"))
                & (
                    truthy(F.col("valid_not_sample_data_flag"))
                    | (F.coalesce(F.col("__valid_direct"), F.lit(0)) == 1)
                    | (F.coalesce(F.col("__valid_version"), F.lit(0)) == 1)
                )
            ).cast("int"),
        )
        .groupBy("model_ver_sid")
        .agg(F.max("__result").alias("__valid_result"), F.max("__data").alias("__valid_data"))
    )

    base = (
        base.join(validation_stats, "model_ver_sid", "left")
        .join(valid_version, "model_ver_sid", "left")
        .fillna({"__valid_result": 0, "__valid_data": 0, "__valid_version": 0})
        .withColumn("validation_not_required", truthy(F.col("model_ver_not_valid_flag")).cast("int"))
        .withColumn(
            "has_validation_result",
            ((F.col("__valid_result") == 1) | required(F.col("model_ver_valid_last_rslt_name"))).cast("int"),
        )
        .withColumn(
            "has_validation_data_path",
            ((F.col("__valid_data") == 1) | (F.col("__valid_version") == 1)).cast("int"),
        )
    )

    # ---------- Мониторинг ----------
    auto_results = read_state(
        "t_montrg_auto_rslt", ["montrg_auto_rslt_sid"],
        ["montrg_auto_sid", "montrg_auto_rslt_name", "montrg_auto_rslt_valid_not_sample_data_flag"],
        as_of, current=False,
    ).filter(required(F.col("montrg_auto_sid")))

    manual_results = read_state(
        "t_montrg_manual_rslt", ["montrg_manual_rslt_sid"],
        ["montrg_manual_sid", "montrg_manual_rslt_name", "montrg_manual_rslt_valid_not_sample_data_flag"],
        as_of, current=False,
    ).filter(required(F.col("montrg_manual_sid")))

    auto_links = read_state(
        "t_model_ver_x_montrg_auto", ["model_ver_sid", "montrg_auto_sid"], [], as_of, current=False
    )
    manual_links = read_state(
        "t_model_ver_x_montrg_manual", ["model_ver_sid", "montrg_manual_sid"], [], as_of, current=False
    )

    auto = monitor_data(samples, auto_results, auto_links, "auto")
    manual = monitor_data(samples, manual_results, manual_links, "manual")

    version_monitor = marker(
        samples, "model_ver_sid",
        required(F.col("model_ver_sid"))
        & norm(F.col("sample_data_type_name")).rlike(r"монитор|стабил")
        & has_location(F.col("sample_data_addtnl_info_txt")),
        "__version_monitor",
    )

    base = (
        base.withColumn("has_monitor_result", required(F.col("model_ver_montrg_last_rslt_name")).cast("int"))
        .join(
            auto,
            (F.col("model_ver_sid") == F.col("__auto_model_ver_sid"))
            & (F.col("model_ver_montrg_last_sid") == F.col("__auto_monitor_sid")),
            "left",
        )
        .join(
            manual,
            (F.col("model_ver_sid") == F.col("__manual_model_ver_sid"))
            & (F.col("model_ver_montrg_last_sid") == F.col("__manual_monitor_sid")),
            "left",
        )
        .join(version_monitor, "model_ver_sid", "left")
        .withColumn(
            "has_monitor_data_path",
            (
                (F.coalesce(F.col("__auto_monitor_data_ok"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__manual_monitor_data_ok"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__version_monitor"), F.lit(0)) == 1)
            ).cast("int"),
        )
    )

    owner_missing = reduce(
        lambda a, b: a | b,
        [~required(F.col(c)) for c in (
            "model_ver_owner_block_name", "model_ver_owner_dprtmt_name", "model_ver_owner_cont_name"
        )],
        F.lit(False),
    )
    dev_missing = reduce(
        lambda a, b: a | b,
        [~required(F.col(c)) for c in (
            "model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_dev_dprtmt_cont_name"
        )],
        F.lit(False),
    )

    base = (
        base
        .withColumn("F01_MODEL_CODE", (~required(F.col("model_code"))).cast("int"))
        .withColumn("F02_MODEL_NAME", (~required(F.col("model_name"))).cast("int"))
        .withColumn("F03_MODEL_CODE_DUP", (F.coalesce(F.col("__code_count"), F.lit(0)) > 1).cast("int"))
        .withColumn("F04_VERSION_NUM", (~required(F.col("model_ver_num"))).cast("int"))
        .withColumn("F05_VERSION_NAME", (~required(F.col("model_ver_name"))).cast("int"))
        .withColumn("F06_DEV_START", (~required(F.col("model_ver_dev_start_fact_dttm"))).cast("int"))
        .withColumn("F07_DEV_END", (~required(F.col("model_ver_dev_end_fact_dttm"))).cast("int"))
        .withColumn("F08_EXPL_START", (~required(F.col("expl_start_dttm"))).cast("int"))
        .withColumn("F09_OWNER", owner_missing.cast("int"))
        .withColumn("F10_DEVELOPER", dev_missing.cast("int"))
        .withColumn(
            "F11_BUSN_PROCESS",
            F.when(is_na(F.col("model_ver_busn_proc_code")), 0)
            .otherwise((~filled(F.col("model_ver_busn_proc_code"))).cast("int")),
        )
        .withColumn("F12_SIGNIFICANCE", (~valid_significance(F.col("model_ver_signfcnt_ctgry_code"))).cast("int"))
        .withColumn("F13_DEV_REPORT", (~required(F.col("model_ver_dev_report_sid"))).cast("int"))
        .withColumn("F14_BITBUCKET_CODE", (~required(F.col("bitbucket_link"))).cast("int"))
        .withColumn(
            "F15_VALIDATION_RESULT",
            ((F.col("validation_not_required") == 0) & (F.col("has_validation_result") == 0)).cast("int"),
        )
        .withColumn(
            "F16_MONITOR_RESULT",
            (required(F.col("model_ver_montrg_last_sid")) & (F.col("has_monitor_result") == 0)).cast("int"),
        )
        .withColumn(
            "F17_VALIDATION_DATA",
            (
                (F.col("validation_not_required") == 0)
                & (F.col("has_validation_result") == 1)
                & (F.col("has_validation_data_path") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F18_MONITOR_DATA",
            (
                required(F.col("model_ver_montrg_last_sid"))
                & (F.col("has_monitor_result") == 1)
                & (F.col("has_monitor_data_path") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F19_SIGNIFICANCE_MISMATCH",
            (
                valid_significance(F.col("model_ver_signfcnt_ctgry_code"))
                & valid_significance(F.col("__anlt_signfcnt"))
                & (norm(F.col("model_ver_signfcnt_ctgry_code")) != norm(F.col("__anlt_signfcnt")))
            ).cast("int"),
        )
        .withColumn("issue_count", reduce(lambda a, b: a + b, [F.col(c) for c in FLAGS], F.lit(0)))
    )

    info = [
        "model_sid", "model_code", "model_name", "model_ver_sid", "model_ver_num", "model_ver_name",
        "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_lvl_name",
        "model_ver_owner_block_name", "model_ver_owner_dprtmt_name", "model_ver_owner_cont_name",
        "model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_dev_dprtmt_cont_name",
        "expl_start_dttm", "bitbucket_link", "issue_count",
    ]
    return base.select(*info, *FLAGS).orderBy("model_sid", "model_ver_sid")


## Excel и запуск

In [ ]:
def write_excel(df, path):
    path = Path(path)
    wb = Workbook(write_only=True)
    ws = wb.create_sheet("Проверка")
    headers = list(df.columns)

    for i in range(1, len(headers) + 1):
        ws.column_dimensions[get_column_letter(i)].width = 24

    row = []
    for h in headers:
        c = WriteOnlyCell(ws, value=h)
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="24435A")
        row.append(c)
    ws.append(row)

    count = 0
    for r in df.toLocalIterator():
        count += 1
        if count > 1_048_575:
            raise RuntimeError("Превышен лимит строк Excel")
        values = []
        for v in r:
            if isinstance(v, (date, datetime)):
                v = v.isoformat()
            elif v is not None and not isinstance(v, (str, int, float, bool)):
                v = str(v)
            if isinstance(v, str):
                v = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", v)[:32767]
            values.append(v)
        ws.append(values)

    if headers:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{count + 1}"
    wb.save(path)


def main():
    report = build_report(AS_OF).cache()
    try:
        print("Версий в эксплуатации:", report.count())
        report.show(20, truncate=False)
        write_excel(report, OUTPUT_FILE)
        print("Сохранено:", OUTPUT_FILE.resolve())
        return report
    except Exception:
        report.unpersist()
        raise


report = main()
